In [0]:
# Cell 1 — Setup Environment

import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

# Hugging Face imports — use Auto classes, not PreTrainedModel
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



In [0]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)



# Cell 2 — Load Sample Super Store dataset
df = spark.read.table("samplesuperstore.bronzedata.orders").toPandas()
df.head()


In [0]:
# Cell 3 — Load embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [0]:
# Cell 4 — Convert rows to embeddings
texts = df.astype(str).apply(lambda row: " ".join(row.values), axis=1).tolist()
embeddings = embedder.encode(texts)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


In [0]:
# Cell 5 — Define retriever
def retrieve(query, top_k=5):
    query_emb = embedder.encode([query])
    distances, indices = index.search(query_emb, top_k)
    return [texts[i] for i in indices[0]]


In [0]:
# Cell 6 — Use Hugging Face GPT-2 for generation
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


In [0]:
# Cell 7 — RAG orchestration
def rag_query(query):
    retrieved = retrieve(query)
    context = "\n".join(retrieved)
    prompt = f"Answer the query based on context:\n{context}\n\nQuery: {query}\nAnswer:"
    output = generator(prompt, max_new_tokens=100)[0]["generated_text"]
    return output

# Example query
print(rag_query("Which category has highest sales?"))
